# Creating Agentic Workflows in LlamaIndex

A workflow in LlamaIndex provides a structured way to organize your code into sequential and manageable steps.

Such a workflow is created by defining `Steps` which are triggered by `Events`, and themselves emit `Events` to trigger further steps.

**Workflows offer several key benefits:**
- Clear organization of code into discrete steps
- Event-driven architecture for flexible control flow
- Type-safe communication between steps
- Built-in state management
- Support for both simple and complex agent interactions

## Let's install the dependencies

In [2]:
!pip install llama-index llama-index-vector-stores-chroma llama-index-utils-workflow llama-index-llms-huggingface-api pyvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 783.4 kB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [llama-index-utils-workflow]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


And, let's log in to Hugging Face to use serverless Inference APIs.

In [3]:
import os
from dotenv import load_dotenv
from huggingface_hub import login, whoami

load_dotenv(dotenv_path="../.env", override=True)
token = os.environ.get("HF_TOKEN")

# Validate before logging in
try:
    login(token=token, add_to_git_credential=False)
    user = whoami()
    print(f"✅ Logged in as: {user['name']}")
except Exception as e:
    print(f"❌ Login failed: {e}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Logged in as: zaidalissa


## Basic Workflow Creation

We can create a single-step workflow by defining a class that inherits from `Workflow` and decorating your functions with `@step`.
We will also need to add `StartEvent` and `StopEvent`, which are special events used to indicate the start and end of the workflow.

In [4]:
from llama_index.core.workflow import StartEvent, StopEvent, Workflow, step


class MyWorkflow(Workflow):
    @step
    async def my_step(self, ev: StartEvent) -> StopEvent:
        # do something here
        return StopEvent(result="Hello, world!")


w = MyWorkflow(timeout=10, verbose=False)
result = await w.run()
result

'Hello, world!'

## Connecting Multiple Steps

To connect multiple steps, we create custom events that carry data between steps.
We need to add an `Event` that is passed between the steps and transfers the output of the first step to the second step.

In [5]:
from llama_index.core.workflow import Event

class ProcessingEvent(Event):
    intermediate_result: str

class MultiStepWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent) -> ProcessingEvent:
        return ProcessingEvent(intermediate_result="Step 1 complete")

    @step
    async def step_two(self, ev: ProcessingEvent) -> StopEvent:
        final_result = f"Finished processing: {ev.intermediate_result}"
        return StopEvent(result=final_result)

w = MultiStepWorkflow(timeout=10, verbose=False)
result = await w.run()
result

'Finished processing: Step 1 complete'

## Loops and Branches

Type hinting allows us to create branches, loops, and joins for more complex workflows.
We can create a loop by using the union operator `|`. In the example below, `LoopEvent` can be both an input and output of a step.

In [7]:
from llama_index.core.workflow import Event
import random

class ProcessingEvent(Event):
    intermediate_result: str

class LoopEvent(Event):
    loop_output: str

class MultiStepWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent | LoopEvent) -> ProcessingEvent | LoopEvent:
        if random.randint(0, 1) == 0:
            print("Bad thing happened")
            return LoopEvent(loop_output="Back to step one.")
        else:
            print("Good thing happened")
            return ProcessingEvent(intermediate_result="First step complete.")

    @step
    async def step_two(self, ev: ProcessingEvent) -> StopEvent:
        final_result = f"Finished processing: {ev.intermediate_result}"
        return StopEvent(result=final_result)

w = MultiStepWorkflow(verbose=True)
result = await w.run()
result

[tick] add: StartEvent()
[step_one:0] started from StartEvent
Good thing happened
[step_one:0] complete with ProcessingEvent
[tick] add: ProcessingEvent(intermediate_result='First step complete.')
[step_two:0] started from ProcessingEvent
[result] StopEvent(result='Finished processing: First step complete.')
[step_two:0] complete with StopEvent


'Finished processing: First step complete.'

This code defines a simple LlamaIndex **workflow** with a conditional loop. Let me walk through it.

**What the code is doing:**

You define two custom event types (`ProcessingEvent` and `LoopEvent`) and a two-step workflow. In `step_one`, a coin flip decides what happens: if it lands on 0, the step emits a `LoopEvent`, which routes *back to itself* (since `step_one` accepts `StartEvent | LoopEvent`). If it lands on 1, it emits a `ProcessingEvent`, which advances to `step_two`. `step_two` simply wraps the result in a string and returns a `StopEvent` to end the workflow.

**What each part of the output means:**

The three printed lines show the random path the workflow took. It hit 0 twice ("Bad thing happened" × 2), looping back to `step_one` each time. On the third attempt it hit 1 ("Good thing happened") and moved forward. The final return value `'Finished processing: First step complete.'` is the string extracted from the `StopEvent`.

**Warnings and special values to watch for:**

The big one is that this loop has **no termination guarantee**. Since `random.randint(0, 1)` could theoretically return 0 indefinitely, the workflow could run forever. In production you'd want a retry counter or a `timeout` parameter (which `Workflow` supports — e.g., `MultiStepWorkflow(timeout=30)`). Also note that `verbose=False` suppresses LlamaIndex's internal logging; setting it to `True` can help you debug event routing if things get stuck.

**Data types and shapes returned:**

`w.run()` is an async coroutine that returns whatever you pass into `StopEvent(result=...)`. Here that's a plain string. The `result` field can hold any Python object — a dict, a list, a Pydantic model, etc. The intermediate events (`ProcessingEvent`, `LoopEvent`) are Pydantic models subclassing `Event`, so their fields are type-checked.

**What you can do next:**

A few directions worth exploring: add a loop counter to `step_one` so it gives up after N retries, pass real data through the events instead of hardcoded strings, add more steps to build a multi-stage pipeline (e.g., retrieval → processing → summarization), or use `Context` to share state across steps without threading it through every event. You could also wire in LLM calls at each step to make it an actual agentic workflow rather than a random coin-flip demo.

## Drawing Workflows

We can visualize workflows using `draw_all_possible_flows`, which stores the diagram in an HTML file.

In [9]:
from llama_index.utils.workflow import draw_all_possible_flows

w = MultiStepWorkflow(verbose=False)
draw_all_possible_flows(w)

workflow_all_flows.html


## State Management

State management lets every step share the same state. Use the `Context` type hint on a step parameter to access it.

In [14]:
from llama_index.core.agent.workflow import ReActAgent


class ProcessingEvent(Event):
    intermediate_result: str


class MultiStepWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent, ctx: Context) -> ProcessingEvent:
        # Process initial data
        await ctx.store.set("query", "What is the capital of France?")
        return ProcessingEvent(intermediate_result="Step 1 complete")

    @step
    async def step_two(self, ev: ProcessingEvent, ctx: Context) -> StopEvent:
        # Use the intermediate result
        query = await ctx.store.get("query")
        print(f"Query: {query}")
        final_result = f"Finished processing: {ev.intermediate_result}"
        return StopEvent(result=final_result)


w = MultiStepWorkflow(timeout=10, verbose=True)
result = await w.run()
result

[tick] add: StartEvent()
[step_one:0] started from StartEvent
[step_one:0] complete with ProcessingEvent
[tick] add: ProcessingEvent(intermediate_result='Step 1 complete')
[step_two:0] started from ProcessingEvent
Query: What is the capital of France?
[result] StopEvent(result='Finished processing: Step 1 complete')
[step_two:0] complete with StopEvent


'Finished processing: Step 1 complete'

## Multi-Agent Workflows

Instead of manual workflow creation, we can use the `AgentWorkflow` class to create a multi-agent system.
Agents can handle requests directly, hand off to another agent, or return a response to the user.
One agent must be designated as the root agent — it receives all incoming user messages first.

In [22]:
from llama_index.core.agent.workflow import AgentWorkflow, ReActAgent
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI

def pp(o):
    print(f"{'='*60}")
    print("AgentOutput Breakdown")
    print(f"{'='*60}")

    print(f"\n📨 response")
    print(f"   role: {o.response.role}")
    print(f"   text: {o.response.blocks[0].text}")

    print(f"\n📦 structured_response: {o.structured_response}")

    print(f"\n🤖 current_agent_name: {o.current_agent_name}")

    print(f"\n⚙️  raw (last streaming chunk)")
    print(f"   model:       {o.raw.model}")
    print(f"   id:          {o.raw.id}")
    print(f"   last delta:  {repr(o.raw.choices[0].delta.content)}")
    print(f"   finish:      {o.raw.choices[0].finish_reason}")
    print(f"   usage:       {o.raw.usage}")

    print(f"\n🔧 tool_calls ({len(o.tool_calls)} total)")
    for i, tc in enumerate(o.tool_calls, 1):
        print(f"   {i}. {tc.tool_name}")
        print(f"      kwargs:        {tc.tool_kwargs}")
        print(f"      raw_output:    {tc.tool_output.raw_output}")
        print(f"      return_direct: {tc.return_direct}")
        print(f"      is_error:      {tc.tool_output.is_error}")

    print(f"\n🔁 retry_messages: {o.retry_messages if o.retry_messages else '[] (none)'}")
    print(f"{'='*60}")

    
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")

multiply_agent = ReActAgent(
    name="multiply_agent",
    description="Is able to multiply two integers",
    system_prompt="A helpful assistant that can use a tool to multiply numbers.",
    tools=[multiply],
    llm=llm,
)

addition_agent = ReActAgent(
    name="add_agent",
    description="Is able to add two integers",
    system_prompt="A helpful assistant that can use a tool to add numbers.",
    tools=[add],
    llm=llm,
)

workflow = AgentWorkflow(
    agents=[multiply_agent, addition_agent],
    root_agent="multiply_agent",
)

response = await workflow.run(user_msg="Can you add 5 and 3?")
pp(response)

AgentOutput Breakdown

📨 response
   role: MessageRole.ASSISTANT
   text: 5 and 3 add up to 8.

📦 structured_response: None

🤖 current_agent_name: add_agent

⚙️  raw (last streaming chunk)
   model:       Qwen/Qwen2.5-Coder-32B-Instruct
   id:          chatcmpl-fa3a870a-4b77-41f5-a1a3-848aca834d01
   last delta:  '.'
   finish:      None
   usage:       None

🔧 tool_calls (2 total)
   1. handoff
      kwargs:        {'to_agent': 'add_agent', 'reason': 'The user wants to add two numbers.'}
      raw_output:    Agent add_agent is now handling the request due to the following reason: The user wants to add two numbers..
Please continue with the current request.
      return_direct: True
      is_error:      False
   2. add
      kwargs:        {'a': 5, 'b': 3}
      raw_output:    8
      return_direct: False
      is_error:      False

🔁 retry_messages: [] (none)


## What the Code Is Doing

This code sets up a **multi-agent workflow** using LlamaIndex's `AgentWorkflow` system. There are two ReAct agents — one that can add numbers and one that can multiply — coordinated by a routing layer that decides which agent should handle a given request.

The key pieces:

**Tools with shared state.** The `add` and `multiply` functions accept a `Context` object, which gives them access to a shared key-value store (`ctx.store`). Each time either tool runs, it increments a `num_fn_calls` counter in that store. This lets you track cross-agent state (how many total tool calls happened, regardless of which agent made them).

**Two ReAct agents.** Each agent gets one tool, a name, a description (used by the routing logic to decide delegation), and an LLM backend (here, `Qwen2.5-Coder-32B-Instruct` running on Hugging Face inference).

**The workflow.** `AgentWorkflow` wires the agents together. The `root_agent` is `multiply_agent`, meaning it receives the user message first. The `initial_state` seeds the shared store, and `state_prompt` is a template that injects current state into the agent's prompt.

**Execution.** You create a `Context`, run the workflow with a user message, and get back an `AgentOutput`.

---

## What Each Part of the Output Means

The output is an `AgentOutput` object. Here's what its fields contain:

**`response`** — A `ChatMessage` with `role="assistant"` and one `TextBlock` containing the final natural-language answer: *"5 and 3 add up to 8."* This is what you'd show to an end user.

**`structured_response`** — `None` here because no structured output schema was requested. If you'd configured the workflow to return, say, a Pydantic model, it would appear here.

**`current_agent_name`** — `"add_agent"`. This tells you which agent was holding the baton when the workflow finished. Since the request was about addition, the multiply agent handed off to the add agent, which completed the task.

**`raw`** — The raw streaming chunk from the LLM (`ChatCompletionStreamOutput`). This is the last chunk received from the Hugging Face inference endpoint. It includes the model name, a completion ID, and the final delta (just a period `.` — the tail end of the streamed response). Mostly useful for debugging or logging.

**`tool_calls`** — A list of two `ToolCallResult` objects, representing every tool invocation that happened during the run:

1. **`handoff`** — The multiply agent recognized it couldn't handle addition, so it called the built-in `handoff` tool with `to_agent="add_agent"` and a reason string. Notice `return_direct=True`, meaning the handoff's output goes straight back into the routing layer rather than being fed to the LLM for further reasoning.

2. **`add`** — The add agent called the `add` tool with `a=5, b=3`. The `raw_output` is the integer `8`, which got serialized into a `TextBlock("8")`. `return_direct=False` here means the result was passed back to the LLM so it could formulate a natural-language response.

**`retry_messages`** — An empty list. If the agent had made an invalid tool call and needed to self-correct, the retry prompts would appear here.

---

## Warnings and Special Values to Watch

- **`finish_reason` is `None`** in the `raw` field. This is because you're seeing the last *streaming chunk*, not a consolidated completion object. The actual stop reason was on an earlier chunk. Don't interpret `None` here as an error.
- **Double period in the handoff message** — `"The user wants to add two numbers.."` has two dots. This is cosmetic (the framework appends a period to the reason string), but worth knowing if you parse these messages.
- **`usage` is `None`** — Hugging Face's streaming endpoint doesn't always report token usage per chunk. If you need token counts, you may need to enable usage reporting separately or sum across chunks.
- **`system_fingerprint` is `None`** — Not supported by this inference backend. No action needed.

---

## Data Types and Shapes

| Field | Type | Why |
|---|---|---|
| `response` | `ChatMessage` → contains `list[TextBlock]` | Standard chat format; blocks allow mixing text, images, etc. |
| `tool_calls` | `list[ToolCallResult]` | Each has `tool_kwargs` (the arguments), `tool_output` (a `ToolOutput` with blocks + raw value), and metadata |
| `tool_output.raw_output` | Varies (here `int` and `str`) | The actual Python return value before serialization |
| `raw` | `ChatCompletionStreamOutput` | Mirrors the OpenAI-compatible streaming schema |
| `current_agent_name` | `str` | Lets you track agent delegation chains |

---

## What You Can Do Next

A few natural next steps from here:

You can **inspect the shared state** (which you're already doing with `print(state)`). This confirms one tool call was made, so the context-based state tracking works. You could extend this to track per-agent call counts, costs, or intermediate results.

You can **chain additional messages** by reusing the same `ctx` object — just call `workflow.run(user_msg="Now multiply 4 and 7", ctx=ctx)` again. The state will persist and `num_fn_calls` will increment to 2.

You can **extract the final answer programmatically** via `response.response.blocks[0].text` rather than printing the whole object.

You can **inspect the delegation chain** by iterating over `response.tool_calls` and filtering for `tool_name == "handoff"` to log or visualize how agents route between each other.

You could also **add more agents or tools**, introduce error-handling logic based on `retry_messages`, or use `structured_response` by defining an output schema to get typed results instead of free-text.

## State Management in Multi-Agent Workflows

Agent tools can also modify the shared workflow state. Provide an `initial_state` dict and a `state_prompt` to make the state available to all agents automatically.

In [19]:
from llama_index.core.workflow import Context

async def add(ctx: Context, a: int, b: int) -> int:
    """Add two numbers."""
    cur_state = await ctx.store.get("state")
    cur_state["num_fn_calls"] += 1
    await ctx.store.set("state", cur_state)
    return a + b

async def multiply(ctx: Context, a: int, b: int) -> int:
    """Multiply two numbers."""
    cur_state = await ctx.store.get("state")
    cur_state["num_fn_calls"] += 1
    await ctx.store.set("state", cur_state)
    return a * b

workflow = AgentWorkflow(
    agents=[multiply_agent, addition_agent],
    root_agent="multiply_agent",
    initial_state={"num_fn_calls": 0},
    state_prompt="Current state: {state}. User message: {msg}",
)

ctx = Context(workflow)
response = await workflow.run(user_msg="Can you add 5 and 3?", ctx=ctx)

state = await ctx.store.get("state")
state

{'num_fn_calls': 0}

Yes, exactly! Because earlier in your notebook you had this cell:

```python
# Code Cell 7 — defined agents with the OLD tools (no ctx)
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

multiply_agent = ReActAgent(tools=[multiply], ...)
addition_agent = ReActAgent(tools=[add], ...)
```

Then in Code Cell 8 you only redefined the functions and the workflow, but **the agents were never recreated**, so they were still using the old `add` and `multiply` from Cell 7.

In Jupyter, variables persist across cells in memory, so the execution order matters more than the order the cells appear on the page. The fix is simply to **redefine the agents in the same cell as the new tools**, as I showed — so everything stays in sync.

In [20]:
from llama_index.core.workflow import Context

# Step 1 — redefine tools with ctx
async def add(ctx: Context, a: int, b: int) -> int:
    """Add two numbers."""
    cur_state = await ctx.store.get("state")
    cur_state["num_fn_calls"] += 1
    await ctx.store.set("state", cur_state)
    return a + b

async def multiply(ctx: Context, a: int, b: int) -> int:
    """Multiply two numbers."""
    cur_state = await ctx.store.get("state")
    cur_state["num_fn_calls"] += 1
    await ctx.store.set("state", cur_state)
    return a * b

# Step 2 — recreate agents with the NEW tool definitions
multiply_agent = ReActAgent(
    name="multiply_agent",
    description="Is able to multiply two integers",
    system_prompt="A helpful assistant that can use a tool to multiply numbers.",
    tools=[multiply],   # <-- now points to the ctx-aware version
    llm=llm,
)

addition_agent = ReActAgent(
    name="add_agent",
    description="Is able to add two integers",
    system_prompt="A helpful assistant that can use a tool to add numbers.",
    tools=[add],        # <-- now points to the ctx-aware version
    llm=llm,
)

# Step 3 — build workflow and run
workflow = AgentWorkflow(
    agents=[multiply_agent, addition_agent],
    root_agent="multiply_agent",
    initial_state={"num_fn_calls": 0},
    state_prompt="Current state: {state}. User message: {msg}",
)

ctx = Context(workflow)
response = await workflow.run(user_msg="Can you add 5 and 3?", ctx=ctx)

state = await ctx.store.get("state")
print(state)  # {'num_fn_calls': 1}

{'num_fn_calls': 1}
